# Error predictor: mean alone vs mean + uncertainty (rebuttal, earlier pooled version)

Superseded by `error_predictor_per_gene.ipynb` (one forest per gene, Table 4 withholding score), which is the reported result. Kept for the record.

Reads `<run-dir>/error_predictor/fold{j}.parquet` from `snake/scripts/canals/error_predictor_precompute.py` (one job per held-out donor, launcher `launch_error_predictor_precompute.sh` with `POOLED=1`).

- A: random forest, rank(|err|) ~ rank(mu). B: rank(|err|) ~ rank(mu) + rank(u). Ranks within (gene, slide), canal 2 `lin_all`.
- Everything from fold j's network: fitted on its errors on the validation donor (random search inside it, split by genes), scored on its errors on the held-out donor. The held-out donor appears nowhere before the test.
- Score: per (gene, slide) Spearman(predicted, |err|) -> n >= 60 -> (gene, donor) weighted by n -> mean over genes -> one point per donor -> Student-t 95% CI over donors. B - A is paired per donor.
- Sanity: Poisson (u = sqrt(mu)) and gene-homoscedastic Gaussian (u constant in a group) must give B = A.

In [1]:
import sys
import json
from pathlib import Path
import numpy as np
import pandas as pd

PROJ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").is_dir())
sys.path.append(str(PROJ))
from src.evaluation.benchmark_summary import donor_lopo_stats
RUN_ID = "pr_classic3"
RESULTS = PROJ / "results"
DATASETS = ["10x_TuPro", "TLS", "LUNG_CANCER"]
RADIUS = "r75"
LAMBDA_DIR = "lambda_0.0"  # Gaussian heads at lambda 0.0
SUBDIR = "error_predictor"
MODELS = ["homoscedastic", "heteroscedastic", "spot_homoscedastic",
          "poisson_cp", "negativebinomial_hetero_noanchor_cp"]
N_PLUS_MIN = 60


def donor_value(frame, col):
    """One held-out donor: (gene) n-weighted mean over its slides -> mean over genes."""
    g = frame.assign(w=frame[col] * frame["n"]).groupby("gene")[["w", "n"]].sum()
    return float((g["w"] / g["n"]).mean())


rows, per_donor = [], []
for ds in DATASETS:
    for m in MODELS:
        leaf = RESULTS / ds / RUN_ID / f"_{ds}_hvg_top3000" / RADIUS / m
        if (leaf / LAMBDA_DIR).is_dir():
            leaf = leaf / LAMBDA_DIR
        files = sorted((leaf / SUBDIR).glob("fold*.parquet"))
        k_expected = len(json.loads((leaf / "splits.json").read_text())["folds"]) if (leaf / "splits.json").is_file() else None
        pts = {"A": {}, "B": {}, "B-A": {}, "r_mu": {}, "r_u": {}}
        for f in files:
            s = json.loads(f.with_suffix(".json").read_text())
            d = pd.read_parquet(f)
            # ONE population for A, B and B - A: groups where both are defined.
            d = d[(d["n"] >= N_PLUS_MIN) & d["rho_A"].notna() & d["rho_B"].notna()]
            donor = s["test_donor"]
            a, b = donor_value(d, "rho_A"), donor_value(d, "rho_B")
            pts["A"][donor], pts["B"][donor], pts["B-A"][donor] = a, b, b - a
            pts["r_mu"][donor] = donor_value(d.dropna(subset=["r_mu"]), "r_mu")
            pts["r_u"][donor] = donor_value(d.dropna(subset=["r_u"]), "r_u")
            per_donor.append(dict(dataset=ds, model=m, donor=donor, val_donor=s["val_donor"],
                                  n_val_spots=s["n_val_spots"], A=a, B=b, diff=b - a,
                                  best_A=s["best"]["A"]["config"], best_B=s["best"]["B"]["config"]))
        rec = dict(dataset=ds, model=m, K=len(files), K_expected=k_expected)
        for key, vals in pts.items():
            if vals:
                st = donor_lopo_stats(vals)
                rec[key] = st["mean"]
                rec[f"{key}_ci"] = f"[{st['ci95_mean'][0]:+.4f}, {st['ci95_mean'][1]:+.4f}]"
        if pts["B-A"]:
            rec["max|B-A|"] = max(abs(v) for v in pts["B-A"].values())
        rows.append(rec)

RES = pd.DataFrame(rows)
PD = pd.DataFrame(per_donor)
pd.set_option("display.width", 250, "display.max_columns", 30, "display.max_colwidth", 80)

print("Spearman(predicted, |err|) per (gene, slide), held-out donor; mean over donors + Student-t 95% CI")
print("A = rank(mu) only, B = rank(mu) + rank(u); r_mu / r_u = Spearman(mu, |err|) / Spearman(u, |err|) for reference\n")
cols = ["dataset", "model", "K", "K_expected", "A", "A_ci", "B", "B_ci", "B-A", "B-A_ci", "max|B-A|", "r_mu", "r_u"]
print(RES.reindex(columns=cols).to_string(index=False, float_format=lambda v: f"{v:+.4f}"))
print("\nPer donor (forests fitted on val_donor):")
print(PD[["dataset", "model", "donor", "val_donor", "n_val_spots", "A", "B", "diff"]].to_string(index=False, float_format=lambda v: f"{v:+.4f}"))
print("\nChosen hyperparameters (random search inside the validation donor):")
print(PD[["dataset", "model", "donor", "best_A", "best_B"]].to_string(index=False))

Spearman(predicted, |err|) per (gene, slide), held-out donor; mean over donors + Student-t 95% CI
A = rank(mu) only, B = rank(mu) + rank(u); r_mu / r_u = Spearman(mu, |err|) / Spearman(u, |err|) for reference

    dataset                               model  K  K_expected       A               A_ci       B               B_ci     B-A             B-A_ci  max|B-A|    r_mu     r_u
  10x_TuPro                       homoscedastic  7           7 +0.2895 [+0.2219, +0.3570] +0.2890 [+0.2213, +0.3566] -0.0005 [-0.0019, +0.0009]   +0.0038 +0.2872     NaN
  10x_TuPro                     heteroscedastic  7           7 +0.3130 [+0.2515, +0.3744] +0.3166 [+0.2613, +0.3719] +0.0036 [-0.0078, +0.0150]   +0.0269 +0.3131 +0.1973
  10x_TuPro                  spot_homoscedastic  7           7 +0.3066 [+0.2423, +0.3709] +0.3043 [+0.2453, +0.3632] -0.0023 [-0.0099, +0.0052]   +0.0147 +0.3054 +0.0572
  10x_TuPro                          poisson_cp  7           7 +0.4923 [+0.4357, +0.5489] +0.4923 [+0.4357, +0